# Ablaciones ECoG (BCI IV-4) — ejecución y monitoreo en Colab

Campaña de **135 entrenamientos**: 9 configuraciones × 3 sujetos × 5 folds. P2 y P4 se verifican por equivalencia con P1 y no se entrenan. La especificación completa está en `docs/plan_experimental.md`.

**Antes de empezar**
1. *Entorno de ejecución → Cambiar tipo de entorno → GPU*. Usa la misma clase de GPU en toda la campaña: TF32 está permitido y cambia ligeramente la aritmética entre GPUs.
2. Sube `sub1_comp.mat`, `sub2_comp.mat` y `sub3_comp.mat` a una carpeta de Google Drive (`DATA_DIR`).
3. Si el repositorio de GitHub es privado, guarda un token de solo lectura en *Secrets* (🔑) con el nombre `GITHUB_TOKEN`.

**Flujo:** ejecuta las celdas en orden. La celda 7 lanza la campaña **en segundo plano**: puedes detener o reejecutar el monitor (celda 8) sin afectar el entrenamiento. Los resultados se escriben en Drive (`RESULTS_DIR`).

**Si Colab se desconecta:** vuelve a ejecutar las celdas 1–3 y luego la 7. Las corridas completadas se saltan, la que quedó a medias se reentrena desde cero y las fallidas se conservan para diagnóstico.

**Varias sesiones en paralelo:** cada sesión debe usar `SUBJECTS`/`FOLDS` disjuntos sobre el mismo `RESULTS_DIR`, por ejemplo una sesión por sujeto.

## 1. GPU y Google Drive

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount("/content/drive")

## 2. Configuración

In [ ]:
GIT_URL = "https://github.com/<usuario>/ecog-architecture-ablations.git"  # ← tu repositorio
GIT_REF = "main"        # recomendado: una etiqueta fija para toda la campaña, p. ej. "campana-v1"
REPO_DIR = "/content/ecog-architecture-ablations"

DRIVE_ROOT = "/content/drive/MyDrive"
DATA_DIR = f"{DRIVE_ROOT}/BCICIV_4_mat"             # contiene sub1_comp.mat, sub2_comp.mat, sub3_comp.mat
RESULTS_DIR = f"{DRIVE_ROOT}/ecog-ablations-results"

SUBJECTS = ["S1", "S2", "S3"]
FOLDS = ["B1", "B2", "B3", "B4", "B5"]
VARIANTS = None          # None = las 9 configuraciones; o p. ej. ["P0", "A3"]

import os
missing = [f for f in ("sub1_comp.mat", "sub2_comp.mat", "sub3_comp.mat") if not os.path.exists(f"{DATA_DIR}/{f}")]
print("datos OK" if not missing else f"faltan en {DATA_DIR}: {missing}")
os.makedirs(RESULTS_DIR, exist_ok=True)

## 3. Código (GitHub) y dependencias
Clona o actualiza el repositorio. El token nunca se imprime ni se guarda en la URL del remoto.

In [ ]:
import base64, subprocess

try:
    from google.colab import userdata
    TOKEN = userdata.get("GITHUB_TOKEN")
except Exception:
    TOKEN = None

def git(*args, cwd=None):
    cmd = ["git"]
    if TOKEN:
        basic = base64.b64encode(f"x-access-token:{TOKEN}".encode()).decode()
        cmd += ["-c", f"http.extraHeader=Authorization: Basic {basic}"]
    out = subprocess.run(cmd + list(args), cwd=cwd, capture_output=True, text=True)
    if out.returncode != 0:
        message = out.stderr.replace(TOKEN, "***") if TOKEN else out.stderr
        raise RuntimeError(f"git {args[0]} falló: {message[-800:]}")
    return out.stdout.strip()

if not os.path.exists(REPO_DIR):
    git("clone", "--quiet", GIT_URL, REPO_DIR)
git("fetch", "--quiet", "--tags", "origin", cwd=REPO_DIR)
git("checkout", "--quiet", GIT_REF, cwd=REPO_DIR)
if git("rev-parse", "--abbrev-ref", "HEAD", cwd=REPO_DIR) != "HEAD":   # rama (no etiqueta): actualizar
    git("pull", "--quiet", "--ff-only", cwd=REPO_DIR)
print("commit:", git("log", "-1", "--format=%h %s (%ci)", cwd=REPO_DIR))

!pip install -q "mne>=1.6"

import sys
sys.path.insert(0, f"{REPO_DIR}/src")
from ecog_ablations.config import load_config
from ecog_ablations import reporting
CFG = load_config(f"{REPO_DIR}/configs/experiment.yaml")
print("configuración:", CFG["_path"], "| retardo", CFG["preprocessing"]["delay_ms"], "ms | TF32", CFG["runtime"]["allow_tf32"])

## 4. Comprobaciones mínimas (`pytest`)
Incluyen la lectura de los tres `.mat`, la equivalencia de la DSP con MNE y las 6 arquitecturas.

In [ ]:
!cd {REPO_DIR} && ECOG_DATA_DIR="{DATA_DIR}" python -m pytest -q 2>&1 | tail -5

## 5. Control de calidad previo
Canales excluidos, fase del guante, recortes por canal y fold, señal cruda frente a limpia y respuesta de los filtros.

In [ ]:
!cd {REPO_DIR} && python scripts/run_experiments.py --quality --data-dir "{DATA_DIR}" --results-dir "{RESULTS_DIR}" 2>&1 | tail -4

import pandas as pd
from IPython.display import display
Q = f"{RESULTS_DIR}/quality"
display(pd.read_csv(f"{Q}/channels.csv"))
display(pd.read_csv(f"{Q}/glove_hold.csv"))
display(pd.read_csv(f"{Q}/finite_and_clipping.csv"))
reporting.show_images([f"{Q}/figures/filter_responses.png"]
                      + [f"{Q}/figures/{s}_{kind}.png" for s in SUBJECTS for kind in ("raw_vs_clean", "clip_fraction")])

## 6. Smoke real (1 época × 2 batches, S1/B1)
Recorre la cadena completa con datos reales en `RESULTS_DIR/_smoke` (separado de la campaña) y estima el tiempo de cómputo. El primer batch incluye el calentamiento de CUDA, así que la estimación es conservadora.

In [ ]:
!cd {REPO_DIR} && python scripts/run_experiments.py --smoke --data-dir "{DATA_DIR}" --results-dir "{RESULTS_DIR}" 2>&1 | grep -v Warning | tail -25

import glob, json
rows = []
for path in glob.glob(f"{RESULTS_DIR}/_smoke/runs/*/S1/B1/history.csv"):
    h = pd.read_csv(path)
    rows.append({"variante": path.split("/")[-4], "s_por_batch": h.train_seconds.iloc[-1] / h.n_batches.iloc[-1],
                 "s_validación": h.val_seconds.iloc[-1]})
est = pd.DataFrame(rows).set_index("variante").sort_index()
batches = 29500 // CFG["training"]["batch_size"]
est["min_por_fit"] = CFG["training"]["epochs"] * (batches * est.s_por_batch + est["s_validación"]) / 60
display(est.round(3))
feat = [json.load(open(p))["compute_seconds"] for p in glob.glob(f"{RESULTS_DIR}/_smoke/quality/features/*.json")]
hours = (est.min_por_fit.mean() * 135 + 90 * sum(feat) / max(len(feat), 1) / 60) / 60
print(f"campaña completa estimada: ~{hours:.1f} h de GPU (orientativo)")

## 7. Lanzar la campaña en segundo plano
El log queda en `RESULTS_DIR/logs/`. Si ya hay un runner vivo en esta máquina, el nuevo se niega a arrancar.

In [ ]:
import time
os.makedirs(f"{RESULTS_DIR}/logs", exist_ok=True)
LOG_PATH = f"{RESULTS_DIR}/logs/runner_{time.strftime('%Y%m%d_%H%M%S')}.log"
args = [sys.executable, "-u", f"{REPO_DIR}/scripts/run_experiments.py", "--data-dir", DATA_DIR,
        "--results-dir", RESULTS_DIR, "--subjects", *SUBJECTS, "--folds", *FOLDS]
if VARIANTS:
    args += ["--variants", *VARIANTS]
PROC = subprocess.Popen(args, stdout=open(LOG_PATH, "a"), stderr=subprocess.STDOUT, cwd=REPO_DIR,
                        start_new_session=True)
time.sleep(20)
print("PID", PROC.pid, "| estado:", "en marcha" if PROC.poll() is None else f"terminó (código {PROC.returncode})")
print(open(LOG_PATH).read()[-2000:])

## 8. Monitor (reejecutable)
Se refresca cada `refresh` segundos y muestra el progreso, la ETA, la curva en vivo de la corrida activa, los resultados parciales por variante y sujeto, las deltas apareadas parciales y las últimas líneas del log. **Detener esta celda no detiene el entrenamiento.**

In [ ]:
reporting.monitor(RESULTS_DIR, CFG, SUBJECTS, FOLDS, VARIANTS, refresh=60)

## 9. Detener la campaña
Envía SIGINT al runner: la corrida en curso queda `interrupted` y se reentrenará al relanzar (celda 7).

In [ ]:
import glob, json, signal, socket
for path in glob.glob(f"{RESULTS_DIR}/runners/{socket.gethostname()}-*.json"):
    pid = json.load(open(path))["pid"]
    try:
        os.killpg(os.getpgid(pid), signal.SIGINT)
        print("SIGINT enviado al runner", pid)
    except ProcessLookupError:
        print("el runner", pid, "ya no existe")

## 10. Reporte
Genera `RESULTS_DIR/summary/` (CSV y figuras) con las corridas disponibles. Se puede ejecutar a mitad de campaña.

In [ ]:
!cd {REPO_DIR} && python scripts/make_report.py --results-dir "{RESULTS_DIR}" 2>&1 | tail -20
import pandas as pd
from IPython.display import display
S = f"{RESULTS_DIR}/summary"
_, deltas = reporting.paired_deltas(reporting.collect_runs(RESULTS_DIR, CFG), CFG)
if not deltas.empty:
    display(deltas[deltas.subject == "global"].round(4))
equivalence = reporting.collect_equivalence(RESULTS_DIR)
if not equivalence.empty:
    display(equivalence.groupby(["preprocessing", "reference"])[["rel_max_diff", "corr", "equivalent"]]
            .agg({"rel_max_diff": "max", "corr": "min", "equivalent": "all"}))
reporting.show_images([f"{S}/figures/{n}.png" for n in
                       ("pearson_by_variant", "paired_deltas", "pearson_per_finger", "training_curves", "example_predictions_P0")])

## 11. Al terminar la sesión
Fuerza la sincronización de Drive antes de cerrar (Colab escribe en caché).

In [ ]:
drive.flush_and_unmount()